# GPT，从零开始，用代码讲清楚 - 练习

关于 [gpt, from scratch 视频](https://www.youtube.com/watch?v=kCc8FmEb1nY) 中练习的笔记。

1. 在 YouTube 上观看 [gpt, from scratch 视频](https://www.youtube.com/watch?v=kCc8FmEb1nY)
2. 然后回来解决这些练习来提升自己 :)

我*强烈*建议在配备 GPU 的机器上完成这些练习。

<style>
/* Keep notebook content printable without horizontal clipping. */
.jp-OutputArea-output img,
.jp-RenderedImage img,
img {
  max-width: 100% !important;
  height: auto !important;
}

.jp-Cell,
.jp-InputArea,
.jp-OutputArea-output,
.jp-RenderedMarkdown,
.text_cell_render,
.rendered_html {
  overflow-wrap: anywhere !important;
  word-break: break-word !important;
}

.jp-InputArea-editor pre,
.jp-RenderedText pre,
.jp-OutputArea-output pre,
.jp-OutputArea-output code,
.highlight pre,
.input_area pre,
.output_area pre,
.output_subarea pre,
.output_text pre,
.output_stream pre,
pre,
code {
  white-space: pre-wrap !important;
  overflow-wrap: anywhere !important;
  word-break: break-word !important;
}

.rendered_html table,
.jp-RenderedHTMLCommon table {
  max-width: 100% !important;
}
</style>


In [5]:
import torch
import random
import torch.nn as nn
from tqdm import tqdm
from torch.nn import functional as F

## 练习 1 - $n$ 维张量掌握挑战

**目标：** 将 `Head` 和 `MultiHeadAttention` 合并为一个类，并行处理所有头，<br>
将头视为另一个批次维度（答案也可以在 [nanoGPT](https://github.com/karpathy/nanoGPT) 中找到）。

让我们看看我们要处理的内容：

In [2]:
block_size = 256 # What is the maximum context length for predictions?
dropout = 0.2    # Dropout probability
n_embd = 384     # Number of hidden units in the Transformer (384/6 = 64 dimensions per head)

In [3]:
class Head(nn.Module):
    """ one head of self-attention """
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        # Register a buffer so that it is not a parameter of the model
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape   # Batch size, block size, vocab size (each token is a vector of size 32)
        k = self.key(x)   # (B,T,C) -> (B,T, head_size)
        q = self.query(x) # (B,T,C) -> (B,T, head_size)
        # Compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * C**-0.5                       # (B, T, head_size) @ (B, head_size, T) = (B, T, T) (T is the block_size)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # Masking all values in wei where tril == 0 with -inf
        wei = F.softmax(wei, dim=-1)                                 # (B, T, T)
        wei = self.dropout(wei)
        # Weighted aggregation of the values
        v = self.value(x) # (B, T, C) -> (B, T, head_size)
        out = wei @ v     # (B, T, T) @ (B, T, head_size) = (B, T, head_size)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)]) # Create num_heads many heads
        self.proj = nn.Linear(n_embd, n_embd)                                   # Projecting back to n_embd dimensions (the original size of the input, because we use residual connections)
        self.dropout = nn.Dropout(dropout)                                      # Dropout layer for regularization

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1) # Concatenate the outputs of all heads
        out = self.dropout(self.proj(out))                  # Project back to n_embd dimensions (because we use residual connections) and apply dropout
        return out

从视频中复制粘贴的部分到此结束，开始干活吧！

In [4]:
# TODO: Merge the two classes into one

我现在将其集成到了源自视频的 GPT 实现中，并首先在 `tiny-shakespeare.txt` 数据集上运行它，以验证实现并生成后续练习所需的基线：

In [ ]:
# TODO: Integrate the combined class from above into the model
# TODO: Verify that your new model works by training it on tiny-shakespeare.txt (you'll need the training loss info and results later)

## 练习 2 - 数学掌握

**目标：** 在你选择的数据集上训练 GPT！还有什么其他数据让它唠叨起来会很有趣？<br>
如果你喜欢，这里有一个有趣的高级建议：训练一个 GPT 来做两个数字的加法，即 $a+b=c$。一旦你有了这个，swole doge 项目：在 GPT 中构建一个计算器克隆，支持所有 $+-*/$。<br>
- 你可能会发现以相反顺序预测 $c$ 的数字是有帮助的，因为典型的加法算法（你希望它学会的）也是从右到左进行的。
- 你可能想要修改数据加载器，让它直接提供随机问题，并跳过 `train.bin`、`val.bin` 的生成。<br>
- 你可能想要在 $a+b$ 的输入位置屏蔽损失，这些位置只是用目标中的 $y=-1$ 来指定问题（参见 CrossEntropyLoss ignore_index）。你的 Transformer 学会加法了吗？一旦你有了这个，swole doge 项目：在 GPT 中构建一个计算器克隆，支持所有 $+-*/$。

**这不是一个简单的问题。** 但是，[GPT 可以在没有计算器的情况下解决数学问题](https://arxiv.org/abs/2309.03241)。<br>
你可能需要 [思维链](https://arxiv.org/abs/2412.14135) 以及其他 [稍微高级一些的架构](https://github.com/deepseek-ai/DeepSeek-V3/blob/main/DeepSeek_V3.pdf) 跟踪，但不要想太多。

In [ ]:
# TODO: Train a model on mathematical expressions so that (some) generated expressions are valid

## 练习 3 - 微调会更好吗？

**目标：** 找一个非常大的数据集，大到你看不到训练损失和验证损失之间的差距。<br>
在这个数据上预训练 transformer。然后，用该模型初始化，并以较少的步数和较低的学习率在 `tiny shakespeare` 上微调。<br>你能通过大规模预训练获得更低的验证损失吗？

In [ ]:
# TODO: Train a model on a text dataset bigger than tiny-shakespeare.txt
# TODO: Use this now pre-trained model to (lightly) fine-tune on tiny-shakespeare.txt
# TODO: Compare the losses and generated text of the fine-tuned model and the model trained from scratch on tiny-shakespeare.txt

## 练习 4 - 阅读并实现

**目标：** 阅读一些 transformer 论文并实现一个人们似乎在使用的额外功能或改动。<br>
它能提升你的 GPT 的性能吗？

In [ ]:
# TODO: The stage is yours! Add any popular model feature and see how it goes!